# Regressing cheddar taste on chemical composition

The cheddar dataset records a panel-tasting score for 30 samples of mature cheddar alongside three chemical measurements that vary with maturation: free acetic acid concentration, hydrogen sulfide, and lactic acid. The question is the standard regression question: how much of the variation in taste can the chemistry explain, and which of the three predictors carries the most weight?

**Data.** `cheddar.csv` from [openmv.net](https://openmv.net/info/cheddar). 30 rows, four numeric columns.

**What we do.** Fit simple linear regression of taste on hydrogen sulfide on its own, then a multiple linear regression on all three predictors, read the coefficient table, and check the residual diagnostics that come with the `OLS` estimator.

**Adapted from** the *Least squares modelling* chapter of the [Process Improvement using Data](https://learnche.org/pid) book (CC BY-SA 4.0).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from process_improve.regression.methods import OLS

## Load the data

openmv CSVs sometimes have an unnamed index column on the left and sometimes do not. We load with `index_col=0` and fall back to a no-index read if the resulting frame has fewer columns than expected.

In [ ]:
url = "https://openmv.net/file/cheddar.csv"
cheddar = pd.read_csv(url, index_col=0)
if cheddar.shape[1] < 4:
    cheddar = pd.read_csv(url)
cheddar.columns = [c.strip() for c in cheddar.columns]
print(f"Shape: {cheddar.shape}")
print(f"Columns: {list(cheddar.columns)}")
cheddar.head()

In [ ]:
# Identify columns case-insensitively so the notebook is robust to small
# naming differences between revisions of the dataset.
cols = {c.lower(): c for c in cheddar.columns}
y_col = cols["taste"]
h2s_col = cols["h2s"]
predictor_cols = [cols["acetic"], cols["h2s"], cols["lactic"]]
y = cheddar[y_col].astype(float)
print(f"Response: {y_col}; Predictors: {predictor_cols}")

## Simple linear regression

Hydrogen sulfide is the predictor that pid-book singles out as the strongest individual driver. Fitting taste on `H2S` alone is a useful sanity check; we expect a positive slope (more mature cheese smells stronger and tastes more) and a comfortable signal-to-noise ratio.

In [ ]:
model_simple = OLS().fit(cheddar[[h2s_col]], y)
print(
    f"intercept = {model_simple.intercept_:.2f}, "
    f"slope = {model_simple.coefficients_[0]:.2f}, "
    f"R2 = {model_simple.r2_:.3f}"
)
print(f"t-value on slope = {model_simple.t_values_[0]:.2f}, p-value = {model_simple.p_values_[0]:.4f}")

In [ ]:
# Scatter plot with fitted line and 95 percent prediction interval band.
x_grid = model_simple.pi_range_[:, 0]
lo = model_simple.pi_range_[:, 1]
hi = model_simple.pi_range_[:, 2]
y_hat = model_simple.intercept_ + model_simple.coefficients_[0] * x_grid

_fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(cheddar[h2s_col], y, s=30, color="#1f77b4", label="observed")
ax.plot(x_grid, y_hat, color="k", linewidth=1.2, label="fit")
ax.fill_between(x_grid, lo, hi, color="orange", alpha=0.25, label="95% prediction band")
ax.set_xlabel(h2s_col)
ax.set_ylabel(y_col)
ax.set_title("Cheddar taste vs hydrogen sulfide")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

## Multiple linear regression

Adding the other two predictors gives the model a chance to explain more of the variation, but tells us less about which predictor matters most: collinearity between the three chemistry variables means the individual coefficient estimates trade against each other.

In [ ]:
model_full = OLS().fit(cheddar[predictor_cols], y)
summary = pd.DataFrame(
    {
        "coefficient": model_full.coefficients_,
        "std_error": model_full.standard_errors_,
        "t_value": model_full.t_values_,
        "p_value": model_full.p_values_,
    },
    index=predictor_cols,
)
summary.loc["(intercept)"] = [
    model_full.intercept_,
    model_full.standard_error_intercept_,
    model_full.t_value_intercept_,
    model_full.p_value_intercept_,
]
print(
    f"R2 = {model_full.r2_:.3f}, adjusted R2 = {model_full.adj_r2_:.3f}, "
    f"F = {model_full.f_statistic_:.2f} (p = {model_full.f_pvalue_:.4f})"
)
summary.round(3)

## Residual diagnostics

Three quick plots tell you most of what you need to know about whether the linear model is appropriate:

- **Residuals vs fitted**: should be a horizontal cloud around zero; a U-shape means a curvature term is missing, a fan shape means the variance grows with the fitted value.
- **Residual histogram or Q-Q plot**: should look approximately normal if you want to trust the t-statistics.
- **Residuals vs sample order** (if applicable): non-random patterns suggest the rows are not independent.

In [ ]:
fitted = model_full.fitted_values_
resid = model_full.residuals_
resid_clean = resid[np.isfinite(resid)]

_fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].scatter(fitted, resid_clean, s=25, color="#1f77b4")
axes[0].axhline(0, color="k", linewidth=0.8)
axes[0].set_xlabel("Fitted")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs fitted")

axes[1].hist(resid_clean, bins=10, color="#1f77b4", edgecolor="k")
axes[1].set_xlabel("Residual")
axes[1].set_title("Residual distribution")

stats.probplot(resid_clean, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q plot")
plt.tight_layout()
plt.show()

## What to try next

- Drop the predictor with the largest p-value and re-fit; compare the coefficient estimates of the two retained predictors. They should change because of the collinearity between the three chemistry variables.
- Try `OLS(fit_intercept=False)` to see what an origin-passing fit looks like; the diagnostics shift even though the data does not.
- For correlated predictors, switch to PLS to avoid the variance-inflation problem altogether. See the latent-variable case studies for examples.